# Lip Movement Classification - Assignment 4

This notebook implements:
1. **Data Extraction** - Extract lip landmarks from videos using MediaPipe
2. **Data Preparation** - Load, pad, encode labels, and split dataset
3. **Classification Models** - GAK+SVM, Shapelet+MLP, LSTM, and additional method

In [1]:
# MediaPipe Face Mesh Setup (Tasks API for v0.10.35+)
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path="face_landmarker.task")
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
)
face_landmarker = vision.FaceLandmarker.create_from_options(options)
print("MediaPipe FaceLandmarker initialized successfully")

MediaPipe FaceLandmarker initialized successfully


## 1. Data Extraction

Extract lip landmarks from video files using MediaPipe Face Mesh. Imported from `src/extract_landmarks.py`

In [2]:
import cv2
import numpy as np
from pathlib import Path

# 40 lip landmark indices (inner + outer lips)
LIP_LANDMARKS = [
    # Outer upper lip
    61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291,
    # Outer lower lip
    375, 321, 405, 314, 17, 84, 181, 91, 146,
    # Inner upper lip
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308,
    # Inner lower lip
    324, 318, 402, 317, 14, 87, 178, 88, 95,
]
NUM_LANDMARKS = len(LIP_LANDMARKS)  # 40
FEATURE_DIM = NUM_LANDMARKS * 2  # x, y per landmark = 80 features

print(f"Using {NUM_LANDMARKS} lip landmarks = {FEATURE_DIM} features (x, y coordinates)")

Using 40 lip landmarks = 80 features (x, y coordinates)


In [3]:
def parse_align(align_path):
    """
    Parse a .align file into a list of (start_frame, end_frame, word) tuples.
    Silent segments ('sil') are excluded.
    """
    segments = []
    with open(align_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue
            start, end, word = int(parts[0]), int(parts[1]), parts[2]
            if word == "sil":
                continue
            # Convert from 1/1000 units to actual frame indices
            start_frame = start // 1000
            end_frame = end // 1000
            segments.append((start_frame, end_frame, word))
    return segments

print("Alignment parser defined")

Alignment parser defined


In [4]:
def extract_landmarks_from_video(video_path):
    """
    Run MediaPipe on every frame of the video.
    Returns an array of shape (num_frames, NUM_LANDMARKS * 2),
    and a list of frame indices where detection succeeded.
    Frames where no face is detected are filled with NaN.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    all_landmarks = []
    frame_indices = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # MediaPipe expects RGB
        import mediapipe as mp
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = face_landmarker.detect(mp_image)

        if result.face_landmarks:
            lms = result.face_landmarks[0]  # list of NormalizedLandmark
            coords = []
            for idx in LIP_LANDMARKS:
                coords.extend([lms[idx].x, lms[idx].y])
            all_landmarks.append(coords)
        else:
            # No face detected — fill with NaN so we can handle it later
            all_landmarks.append([np.nan] * FEATURE_DIM)

        frame_indices.append(frame_idx)
        frame_idx += 1

    cap.release()
    return np.array(all_landmarks, dtype=np.float32), np.array(frame_indices)

print("Landmark extraction function defined")

Landmark extraction function defined


In [5]:
def normalize_sequence(seq):
    """
    Normalize lip landmarks per frame relative to the mouth bounding box.
    This removes variation from head distance and position.
    
    seq: shape (T, NUM_LANDMARKS * 2)
    returns: normalized array of same shape
    """
    normalized = seq.copy()
    T = seq.shape[0]

    for t in range(T):
        frame = seq[t].reshape(NUM_LANDMARKS, 2)  # (40, 2)
        if np.any(np.isnan(frame)):
            continue  # leave NaN frames as-is

        x_coords, y_coords = frame[:, 0], frame[:, 1]
        x_min, x_max = x_coords.min(), x_coords.max()
        y_min, y_max = y_coords.min(), y_coords.max()

        w = x_max - x_min if x_max > x_min else 1e-6
        h = y_max - y_min if y_max > y_min else 1e-6

        # Normalize to [0, 1] within the lip bounding box
        norm_x = (x_coords - x_min) / w
        norm_y = (y_coords - y_min) / h

        normalized[t] = np.stack([norm_x, norm_y], axis=1).reshape(-1)

    return normalized

print("Normalization function defined")

Normalization function defined


In [6]:
def process_dataset(data_root, output_root):
    """
    Walk through the dataset and extract .npz files for every word segment.
    
    Expected structure:
        data_root/
          alignments/s1/*.align
          s1/*.mpg
    
    Output structure:
        output_root/
          <word>/
            <video_stem>_<start>_<end>.npz
    """
    data_root = Path(data_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    # Find all speakers (s1, s2, ...)
    speakers = [d for d in (data_root).iterdir() if d.is_dir() and d.name.startswith("s")]

    total_saved = 0
    total_failed = 0

    for speaker_dir in sorted(speakers):
        speaker = speaker_dir.name
        align_dir = data_root / "alignments" / speaker
        video_dir = data_root / speaker

        video_files = sorted(video_dir.glob("*.mpg"))
        print(f"\n[{speaker}] Found {len(video_files)} videos")

        for video_path in video_files:
            stem = video_path.stem  # e.g. "bbaf2n"
            align_path = align_dir / f"{stem}.align"

            if not align_path.exists():
                print(f"  SKIP {stem}: no alignment file")
                continue

            # Parse word segments
            segments = parse_align(align_path)
            if not segments:
                continue

            # Extract landmarks for the whole video (once per video)
            try:
                landmarks, frame_indices = extract_landmarks_from_video(video_path)
            except Exception as e:
                print(f"  ERROR {stem}: {e}")
                total_failed += 1
                continue

            # Slice into word-level sequences and save
            for start_f, end_f, word in segments:
                # Clamp to valid frame range
                start_f = max(0, start_f)
                end_f = min(end_f, len(landmarks))

                if end_f <= start_f:
                    continue

                raw_seq = landmarks[start_f:end_f]          # (T, 80)
                norm_seq = normalize_sequence(raw_seq)       # (T, 80)
                seg_frames = frame_indices[start_f:end_f]   # (T,)

                # Save to output_root/<word>/<stem>_<start>_<end>.npz
                word_dir = output_root / word
                word_dir.mkdir(parents=True, exist_ok=True)
                out_path = word_dir / f"{stem}_{start_f}_{end_f}.npz"

                np.savez(
                    out_path,
                    landmarks=raw_seq,          # raw x,y coordinates
                    normalized=norm_seq,         # normalized sequence
                    frame_indices=seg_frames,    # original frame numbers
                    label=word,                  # word label (str)
                    video_path=str(video_path), # source video
                )
                total_saved += 1

            print(f"  {stem}: {len(segments)} words saved")

    print(f"\nDone. Saved {total_saved} sequences, {total_failed} videos failed.")
    return total_saved, total_failed

In [7]:
# Run data extraction
DATA_ROOT = "archive/data"
OUTPUT_ROOT = "extracted_features"

# Uncomment to run extraction:
# total_saved, total_failed = process_dataset(DATA_ROOT, OUTPUT_ROOT)
print("Data extraction functions loaded from src/extract_landmarks.py")

Data extraction functions loaded from src/extract_landmarks.py


## 2. Data Preparation

Load extracted .npz files, encode labels, pad/truncate sequences, and split into train/test sets. Imported from `src/data_preparation.py`

In [8]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def load_dataset(features_root):
    """
    Walk through extracted_features/<word>/*.npz and load all sequences.
    
    Returns:
        sequences : list of np.ndarray, each shape (T_i, 80) — variable length
        labels    : list of str, word label for each sequence
    """
    features_root = Path(features_root)
    sequences = []
    labels = []

    for word_dir in sorted(features_root.iterdir()):
        if not word_dir.is_dir():
            continue
        word = word_dir.name

        for npz_path in sorted(word_dir.glob("*.npz")):
            data = np.load(npz_path, allow_pickle=True)
            seq = data["normalized"]  # shape (T, 80)

            # Skip sequences that are entirely NaN (failed extraction)
            if np.all(np.isnan(seq)):
                continue

            # Replace any remaining NaN frames with zeros
            seq = np.nan_to_num(seq, nan=0.0)

            sequences.append(seq)
            labels.append(word)

    print(f"Loaded {len(sequences)} sequences across {len(set(labels))} classes")
    return sequences, labels

In [9]:
def analyze_lengths(sequences, plot=True):
    """
    Print length statistics and optionally plot a histogram.
    Use this to choose a good MAX_LEN for padding/truncation.
    """
    lengths = [len(s) for s in sequences]
    lengths = np.array(lengths)

    print(f"Sequence length stats:")
    print(f"  min    : {lengths.min()}")
    print(f"  max    : {lengths.max()}")
    print(f"  mean   : {lengths.mean():.1f}")
    print(f"  median : {np.median(lengths):.1f}")
    print(f"  p90    : {np.percentile(lengths, 90):.1f}")
    print(f"  p95    : {np.percentile(lengths, 95):.1f}")

    if plot:
        plt.figure(figsize=(8, 4))
        plt.hist(lengths, bins=40, edgecolor="black")
        plt.axvline(np.percentile(lengths, 95), color="red", linestyle="--",
                    label=f"95th percentile ({np.percentile(lengths, 95):.0f})")
        plt.xlabel("Sequence length (frames)")
        plt.ylabel("Count")
        plt.title("Distribution of word sequence lengths")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return lengths

In [10]:
def pad_sequences(sequences, max_len):
    """
    Pad sequences shorter than max_len with zeros at the end,
    and truncate sequences longer than max_len from the end.
    
    Args:
        sequences : list of np.ndarray, each shape (T_i, 80)
        max_len   : int, fixed sequence length to use
    
    Returns:
        X : np.ndarray of shape (N, max_len, 80)
    """
    feature_dim = sequences[0].shape[1]  # 80
    X = np.zeros((len(sequences), max_len, feature_dim), dtype=np.float32)

    for i, seq in enumerate(sequences):
        T = len(seq)
        if T >= max_len:
            X[i] = seq[:max_len]       # truncate
        else:
            X[i, :T] = seq             # pad with zeros at the end

    return X

In [11]:
def encode_labels(labels):
    """
    Convert string labels to integer class indices.
    
    Returns:
        y      : np.ndarray of int, shape (N,)
        le     : fitted LabelEncoder (use le.inverse_transform to decode)
    """
    le = LabelEncoder()
    y = le.fit_transform(labels)
    print(f"Classes ({len(le.classes_)}): {list(le.classes_)}")
    return y, le

In [12]:
def split_dataset(X, y, test_size=0.2, random_state=42):
    """
    Stratified train/test split so class distribution is balanced in both sets.
    
    Returns:
        X_train, X_test, y_train, y_test
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,          # preserve class distribution
    )
    print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")
    return X_train, X_test, y_train, y_test

In [13]:
def prepare_data(features_root="extracted_features", test_size=0.2, max_len=None):
    """
    Run the full data preparation pipeline.
    
    Args:
        features_root : path to extracted_features/
        test_size     : fraction of data to use for testing
        max_len       : fixed sequence length. If None, uses the 95th percentile
                        of sequence lengths automatically.
    
    Returns:
        X_train, X_test : np.ndarray, shape (N, max_len, 80)
        y_train, y_test : np.ndarray, shape (N,)
        le              : LabelEncoder (to decode predictions back to words)
        max_len         : the max_len that was used (useful if auto-selected)
    """
    # Load
    sequences, labels = load_dataset(features_root)

    # Analyze lengths and choose max_len
    lengths = analyze_lengths(sequences, plot=True)
    if max_len is None:
        max_len = int(np.percentile(lengths, 95))
        print(f"\nAuto-selected MAX_LEN = {max_len} (95th percentile)")
    else:
        print(f"\nUsing MAX_LEN = {max_len}")

    covered = np.mean(lengths <= max_len) * 100
    print(f"  {covered:.1f}% of sequences fit without truncation")

    # Pad / truncate
    X = pad_sequences(sequences, max_len)
    print(f"\nPadded dataset shape: {X.shape}")  # (N, max_len, 80)

    # Encode labels
    y, le = encode_labels(labels)

    # Split
    X_train, X_test, y_train, y_test = split_dataset(X, y, test_size=test_size)

    return X_train, X_test, y_train, y_test, le, max_len

In [14]:
# Run data preparation
FEATURES_ROOT = "extracted_features"

# Uncomment to run preparation:
# X_train, X_test, y_train, y_test, le, max_len = prepare_data(
#     features_root=FEATURES_ROOT,
#     test_size=0.2,
#     max_len=None,  # auto-select from data
# )
print("Data preparation functions loaded from src/data_preparation.py")

Data preparation functions loaded from src/data_preparation.py


## 3. Classification Models

Implement the following methods with ablation studies:
1. **GAK + SVM** - Global Alignment Kernel with SVM
2. **Shapelet + MLP** - Shapelet transform with PyTorch MLP
3. **LSTM** - PyTorch LSTM
4. **Extra** - Additional method (1D-CNN, DTW-kNN, or Rocket)

In [15]:
# Import classification libraries
import torch
import torch.nn as nn
from tslearn.kernel import GlobalAlignmentKernel
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

print("Classification libraries imported")

ModuleNotFoundError: No module named 'tslearn.kernel'